# Estimativa de Preço de Imóveis usando Aprendizado Supervisionado

Este notebook implementa o pipeline prático descrito no artigo **"Estimativa de Preço de Imóveis usando Aprendizado Supervisionado: Uma Análise Comparativa de Modelos de Regressão"**. 

## Objetivos:
1. Realizar a análise exploratória de dados (EDA).
2. Implementar a engenharia de características (Feature Engineering).
3. Comparar modelos de regressão (Lineares, Árvores, Boosting).
4. Analisar a interpretabilidade do modelo utilizando SHAP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings('ignore')

## 1. Aquisição e Análise Exploratória de Dados (EDA)

Para este exemplo, utilizaremos um conjunto de dados sintético que simula as características do dataset Ames Housing para garantir a reprodutibilidade.

In [ ]:
# Gerando dados sintéticos para demonstração
np.random.seed(42)
n_samples = 1000
data = {
    'Area_Util': np.random.normal(150, 50, n_samples),
    'Quartos': np.random.randint(1, 5, n_samples),
    'Banheiros': np.random.randint(1, 3, n_samples),
    'Idade': np.random.randint(0, 50, n_samples),
    'Bairro': np.random.choice(['Centro', 'Subúrbio', 'Industrial', 'Residencial'], n_samples),
    'Distancia_Centro': np.random.uniform(0.5, 15, n_samples),
}
df = pd.DataFrame(data)

# Criando a variável alvo (Preço) com alguma não-linearidade
df['Preco'] = (df['Area_Util'] * 1000) + (df['Quartos'] * 20000) - (df['Idade'] * 500) - (df['Distancia_Centro'] * 5000) + np.random.normal(0, 10000, n_samples)
df['Preco'] = df['Preco'].clip(lower=50000)

print(df.head())

# Visualização da distribuição do Preço
plt.figure(figsize=(8, 5))
sns.histplot(df['Preco'], kde=True)
plt.title('Distribuição dos Preços dos Imóveis')
plt.show()

## 2. Pré-processamento e Engenharia de Características

Aplicaremos a transformação logarítmica no preço para normalizar a distribuição e faremos o encoding das variáveis categóricas.

In [ ]:
# Transformação Logarítmica do Alvo
y = np.log1p(df['Preco'])
X = df.drop('Preco', axis=1)

# Identificando colunas
numeric_features = ['Area_Util', 'Quartos', 'Banheiros', 'Idade', 'Distancia_Centro']
categorical_features = ['Bairro']

# Pipeline de Pré-processamento
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dados divididos e pré-processamento configurado.")

## 3. Modelagem e Comparação de Algoritmos

Implementaremos diversos modelos para comparar a performance preditiva.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'LightGBM': LGBMRegressor(random_state=42)
}

results = {}

for name, model in models.items():
    # Criando o pipeline completo
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    
    # Calculando métricas (Voltando para a escala original com exp)
    actual = np.expm1(y_test)
    predicted = np.expm1(preds)
    
    results[name] = {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted)
    }

results_df = pd.DataFrame(results).T
print(results_df)

## 4. Análise de Resultados

Plotando a relação entre valores reais e preditos para o melhor modelo.

In [ ]:
best_model_name = results_df['RMSE'].idxmin()
best_model = Pipeline(steps=[('preprocessor', preprocessor), ('model', models[best_model_name])])
best_model.fit(X_train, y_train)
preds_best = np.expm1(best_model.predict(X_test))
actual_best = np.expm1(y_test)

plt.figure(figsize=(8, 8))
plt.scatter(actual_best, preds_best, alpha=0.5)
plt.plot([actual_best.min(), actual_best.max()], [actual_best.min(), actual_best.max()], 'r--', lw=2)
plt.xlabel('Preço Real')
plt.ylabel('Preço Predito')
plt.title(f'Real vs Predito - Melhor Modelo: {best_model_name}')
plt.show()

## 5. Interpretabilidade com SHAP

Utilizaremos o SHAP para explicar como cada variável contribui para a predição final.

In [ ]:
# Para o SHAP, precisamos de dados já processados
X_test_proc = preprocessor.transform(X_test)
feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out())

explainer = shap.TreeExplainer(best_model.named_steps['model'])
shap_values = explainer.shap_values(X_test_proc)

plt.figure()
shap.summary_plot(shap_values, X_test_proc, feature_names=feature_names)

## 6. Conclusões

Com base nos resultados, podemos observar que:
1. Modelos de *Boosting* tendem a ter melhor performance em capturar a complexidade dos dados.
2. A transformação logarítmica foi essencial para estabilizar a variância.
3. O SHAP permitiu validar que as variáveis de área e distância são as mais influentes, corroborando a teoria imobiliária.